In [1]:
# Import libraries

import pandas as pd
import os
from dotenv import load_dotenv
from xgboost import XGBClassifier

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV

)

In [2]:
# Load environment variables

load_dotenv()

device = os.getenv("GPU") or "cpu"

In [3]:
# Load the processed dataset

df = pd.read_csv("cleaned_data_final.csv")

In [4]:
# Separate predictors and target

X = df.drop(columns=["event_adverse"])
y = df["event_adverse"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

In [5]:
# Model setup

model = XGBClassifier(
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    device=device,

)

param_grid = {
    "n_estimators": [30, 150, 300],
    "max_depth": [3, 4, 5],
    "learning_rate": [0.01, 0.05, 0.1],
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="accuracy",
    cv=5,
    verbose=3,
)

In [6]:
# Train the models

grid.fit(X_train, y_train)

Fitting 5 folds for each of 27 candidates, totalling 135 fits


/home/naya/healthcare-1e/.venv/lib/python3.14/site-packages/xgboost/core.py:569: UserWarning: [12:06:18] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[CV 1/5] END learning_rate=0.005, max_depth=3, n_estimators=30;, score=0.859 total time=   0.4s
[CV 2/5] END learning_rate=0.005, max_depth=3, n_estimators=30;, score=0.859 total time=   0.2s
[CV 3/5] END learning_rate=0.005, max_depth=3, n_estimators=30;, score=0.859 total time=   0.2s
[CV 4/5] END learning_rate=0.005, max_depth=3, n_estimators=30;, score=0.859 total time=   0.2s
[CV 5/5] END learning_rate=0.005, max_depth=3, n_estimators=30;, score=0.859 total time=   0.2s
[CV 1/5] END learning_rate=0.005, max_depth=3, n_estimators=300;, score=0.859 total time=   0.5s
[CV 2/5] END learning_rate=0.005, max_depth=3, n_estimators=300;, score=0.859 total time=   0.4s
[CV 3/5] END learning_rate=0.005, max_depth=3, n_estimators=300;, score=0.859 total time=   0.5s
[CV 4/5] END learning_rate=0.005, max_depth=3, n_estimators=300;, score=0.859 total time=   0.4s
[CV 5/5] END learning_rate=0.005, max_depth=3, n_estimators=300;, score=0.859 total time=   0.4s
[CV 1/5] END learning_rate=0.005, m

KeyboardInterrupt: 

In [ ]:
# Show results

results = pd.DataFrame(grid.cv_results_)

table = results[
    [
        "param_n_estimators",
        "param_max_depth",
        "param_learning_rate",
        "mean_test_score",
    ]
].copy()

table = table.rename(columns={
    "param_n_estimators": "n_estimators",
    "param_max_depth": "max_depth",
    "param_learning_rate": "learning_rate",
    "mean_test_score": "cv_score_accuracy",
})

table = table.sort_values(
    "cv_score_accuracy",
    ascending=False
).reset_index(drop=True)

display(
    table.style.format({
        "learning_rate": "{:.3f}",
        "cv_score_accuracy": "{:.6f}",
        "score_std": "{:.4f}",
    })
)